In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
import glob
import os
import re
import seaborn as sns

mpl.rcParams.update({
    "backend": "pgf",

    "pgf.texsystem": "xelatex",

    "text.usetex": True,

    "pgf.preamble": r"""
        \usepackage{fontspec}
        \usepackage{unicode-math}

        \setmainfont{TeX Gyre Termes}
        \setsansfont{TeX Gyre Heros}
        \setmonofont{TeX Gyre Cursor}

        \setmathfont{TeX Gyre Pagella Math}
    """,

    "font.family": "serif",
    "font.size": 8,          # General font size fallback
    "axes.titlesize": 9,     # Subplot title font size
    "axes.labelsize": 8,     # X and Y axis label size
    "xtick.labelsize": 7,    # X tick numbers size
    "ytick.labelsize": 7,    # Y tick numbers size
    "legend.fontsize": 8,    # Legend font size
})

LATEX_TEXTWIDTH_PT = 370.0
PT_TO_INCH = 1.0 / 72.27

COLORS = {
    "light_blue": "#56B4E9",
    "dark_blue": "#0072B2",
    "orange": "#E69F00",
    "green":"#009E73",
    "red": "#D55E00",
    "pink": "#CC79A7",
}

In [4]:
def process_client_files(data_dir="../results", cutoff_mins=0):
    files = sorted(glob.glob(data_dir + "/client_node-*.log"))
    if not files:
        raise FileNotFoundError("No matching client log files found.")

    raw_frames = []
    for file in files:
        df = pd.read_csv(file)
        df["request_time"] = pd.to_datetime(df["request_time"], unit="ms")
        df["response_time"] = pd.to_datetime(df["response_time"], unit="ms")
        df["client"] = os.path.basename(file).replace(".log", "").replace("client_node-", "")
        raw_frames.append(df)
        
    global_min_time = min(df["request_time"].min() for df in raw_frames)
    processed_frames = []

    cutoff_timedelta = pd.to_timedelta(cutoff_mins, unit="m") if cutoff_mins > 0 else None
    
    for df in raw_frames:
        df["request_time_since_start"] = df["request_time"] - global_min_time
        df["response_time_since_start"] = df["response_time"] - global_min_time

        if cutoff_timedelta is not None:
            df = df[df["request_time_since_start"] >= cutoff_timedelta].copy()
            new_start = df["request_time_since_start"].min()
            df["request_time_since_start"] -= new_start
            df["response_time_since_start"] -= new_start

        latency_series = (df["response_time_since_start"] - df["request_time_since_start"]).dt.total_seconds() * 1000
        
        total_req = latency_series.size
        dropped = latency_series.isna().sum()
        drop_pct = (dropped / total_req * 100) if total_req > 0 else 0
        print(f"Client {df['client'].iloc[0]}: {dropped} of {total_req} requests dropped ({drop_pct:.2f}%)")

        df["latency"] = latency_series.astype(float).fillna(np.inf)
        df["accept_status"] = df["accept_status"].fillna("Dropped")
        
        processed_frames.append(df)
        
    return pd.concat(processed_frames, ignore_index=True)

In [5]:
def parse_fast_path_logs(data_dir="../results", cutoff_mins=0):
    files = sorted(glob.glob(data_dir + "/server_node-*.log"))
    
    pattern = re.compile(
        r'^(?P<timestamp>[A-Za-z]{3}\s+\d+\s+\d{2}:\d{2}:\d{2}\.\d+)\s+INFO\s+===\s+Fast Path Statistics\s+===\s*\n'
        r'Total Append Attempts:\s*(?P<total_attempts>\d+)\s*\n'
        r'Fast Path Attempts:\s*(?P<fp_attempts>\d+).*\n'
        r'.*Successes:\s*(?P<successes>\d+).*\n'
        r'.*Errors:\s*(?P<errors>\d+)',
        re.MULTILINE
    )

    all_server_dfs = []

    for file_path in files:
        filename = os.path.basename(file_path)
        server_name = filename.replace("server_node-", "").replace(".log", "")

        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()

        matches = list(pattern.finditer(content))
        valid_matches = matches[cutoff_mins:] if cutoff_mins < len(matches) else []

        records = []
        for m in valid_matches:
            records.append({
                'server': server_name,
                'timestamp': m.group('timestamp'),
                'total_attempts': int(m.group('total_attempts')),
                'fp_attempts': int(m.group('fp_attempts')),
                'successes': int(m.group('successes')),
                'errors': int(m.group('errors'))
            })

        df_server = pd.DataFrame(records)
        count_cols = ['total_attempts', 'fp_attempts', 'successes', 'errors']
        df_server[count_cols] = df_server[count_cols].cumsum()

        all_server_dfs.append(df_server)

    return pd.concat(all_server_dfs, ignore_index=True)

In [6]:
def plot_client_load(data_dir, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    fig_height = fig_width * 1.0 
    bin_size = 0.5
    
    combined_df = process_client_files(data_dir=data_dir)
    
    # Slice a 200-second window from the middle
    original_max_time = combined_df["request_time_since_start"].dt.total_seconds().max()
    mid_time = original_max_time / 2
    start_time = max(0, mid_time - 100)
    end_time = start_time + 200
    mask = (
        (combined_df["request_time_since_start"].dt.total_seconds() >= start_time) & 
        (combined_df["request_time_since_start"].dt.total_seconds() <= end_time)
    )
    combined_df = combined_df[mask].copy()
    combined_df["request_time_since_start"] -= pd.to_timedelta(start_time, unit='s')
    max_time = 200.0
    all_bins = np.arange(0, max_time + bin_size, bin_size)

    def get_resampled(df, bins):
        df = df.copy()
        df["bin"] = (df["request_time_since_start"].dt.total_seconds() / bin_size).pipe(np.floor) * bin_size
        
        resampled = df.groupby("bin").size().reindex(bins, fill_value=0).reset_index(name="req_count")
        resampled["rps"] = resampled["req_count"] / bin_size
        return resampled

    fig, axes = plt.subplots(
        nrows=2, ncols=1, sharex=True, 
        figsize=(fig_width, fig_height), 
        constrained_layout=True
    )
    
    ax_clients = axes[0]
    ax_agg = axes[1]
    client_colors = [COLORS["orange"], COLORS["green"], COLORS["red"], COLORS["light_blue"], COLORS["pink"]]
    
    for i, (client_name, data) in enumerate(combined_df.groupby("client")):
        resampled = get_resampled(data, all_bins)
        latex_label = rf"\texttt{{{client_name.title()}}}"
        
        color = client_colors[i % len(client_colors)]
        
        ax_clients.plot(
            resampled["bin"], resampled["rps"], 
            linewidth=1.2, 
            color=color,
            label=latex_label
        )
        
    ax_clients.set_title(r"\textbf{Client Requests per Second}")
    ax_clients.set_ylabel(r"Requests Per Second")
    ax_clients.grid(True, alpha=0.3)
    ax_clients.set_xlabel("") 
    
    total_resampled = get_resampled(combined_df, all_bins)
    
    ax_agg.plot(
        total_resampled["bin"], total_resampled["rps"], 
        color=COLORS["dark_blue"], 
        linewidth=1.5, 
        label=r"\texttt{Total Aggregate}"
    )
    ax_agg.set_title(r"\textbf{Aggregated Requests per Second}")
    ax_agg.set_xlabel(r"Time (s)")
    ax_agg.set_ylabel(r"Requests Per Second")
    ax_agg.grid(True, alpha=0.3)

    handles_clients, labels_clients = ax_clients.get_legend_handles_labels()
    handles_agg, labels_agg = ax_agg.get_legend_handles_labels()
    
    all_handles = handles_clients + handles_agg
    all_labels = labels_clients + labels_agg

    fig.legend(
        all_handles, all_labels, 
        loc='lower center', 
        bbox_to_anchor=(0.5, 1.01), 
        ncol=3, 
        frameon=True, 
        fontsize="x-small",
        borderaxespad=0.0
    )
    
    fig.savefig(result_file, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


plot_client_load(
    data_dir="../results/old10/normal_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
    result_file="results/client_load_localevents",
) 

Client us-central1: 0 of 3000 requests dropped (0.00%)
Client us-east5: 0 of 3570 requests dropped (0.00%)
Client us-west1: 0 of 3503 requests dropped (0.00%)
Client us-west2: 0 of 3387 requests dropped (0.00%)
Client us-west4: 0 of 3252 requests dropped (0.00%)


In [7]:
def plot_clients_vs_others_load(experiments, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    bin_size = 0.5
    max_time = 200.0
    all_bins = np.arange(0, max_time + bin_size, bin_size)

    def get_resampled(df, bins):
        df = df.copy()
        df["bin"] = (df["request_time_since_start"].dt.total_seconds() / bin_size).pipe(np.floor) * bin_size
        
        resampled = df.groupby("bin").size().reindex(bins, fill_value=0).reset_index(name="req_count")
        resampled["rps"] = resampled["req_count"] / bin_size
        return resampled
        
    plot_data = []

    for exp in experiments:
        data_dir = exp["data_dir"]
        target_clients = exp["clients"]
        
        combined_df = process_client_files(data_dir=data_dir)

        # Slice a 200-second window from the middle (Independent per directory)
        original_max_time = combined_df["request_time_since_start"].dt.total_seconds().max()
        mid_time = original_max_time / 2
        start_time = max(0, mid_time - 100)
        end_time = start_time + 200
        mask = (
            (combined_df["request_time_since_start"].dt.total_seconds() >= start_time) & 
            (combined_df["request_time_since_start"].dt.total_seconds() <= end_time)
        )
        combined_df = combined_df[mask].copy()
        combined_df["request_time_since_start"] -= pd.to_timedelta(start_time, unit='s')
        
        for client_name in target_clients:
            client_data = combined_df[combined_df["client"] == client_name].copy()
            others_data = combined_df[combined_df["client"] != client_name].copy()
            plot_data.append((client_name, client_data, others_data))

    subplot_width = fig_width / 2.0 
    subplot_height = subplot_width * 0.618
    fig_height = subplot_height * 3
    
    fig, axes = plt.subplots(
        nrows=3, 
        ncols=2, 
        figsize=(fig_width, fig_height), 
        sharex=True, 
        sharey=True, 
        constrained_layout=True
    )
    
    status_configs = {
        "FastAccept": {"color": COLORS["pink"], "ymin": 0.00, "ymax": 0.03, "label": "Fast-Path Block"},
        "LeaderAccept": {"color": COLORS["orange"], "ymin": 0.04, "ymax": 0.07, "label": "Regular-Path Block"}
    }

    for idx, (client_name, client_data, others_data) in enumerate(plot_data):
        row, col = divmod(idx, 2)
        ax = axes[row, col]

        client_resampled = get_resampled(client_data, all_bins)
        others_resampled = get_resampled(others_data, all_bins)
        
        ax.plot(others_resampled["bin"], others_resampled["rps"], color=COLORS["red"], linewidth=1, alpha=0.3)
        ax.plot(client_resampled["bin"], client_resampled["rps"], color=COLORS["green"], linewidth=1)
        
        client_sorted = client_data.sort_values("request_time_since_start")
        times = client_sorted["request_time_since_start"].dt.total_seconds().values
        statuses = client_sorted["accept_status"].values
        
        intervals = {"FastAccept": [], "LeaderAccept": []}
        active_status = None
        group_start = None
        group_end = None
        
        for i in range(len(client_sorted)):
            curr_time = times[i]
            curr_status = statuses[i]
            
            if active_status is not None:
                gap = curr_time - times[i-1]
                opposing_status = "LeaderAccept" if active_status == "FastAccept" else "FastAccept"
                
                if curr_status == opposing_status or gap >= 1.0:
                    intervals[active_status].append((group_start, group_end))
                    
                    if curr_status in ["FastAccept", "LeaderAccept"]:
                        active_status = curr_status
                        group_start = curr_time
                        group_end = curr_time
                    else:
                        active_status = None
                else:
                    group_end = curr_time
            else:
                if curr_status in ["FastAccept", "LeaderAccept"]:
                    active_status = curr_status
                    group_start = curr_time
                    group_end = curr_time
                    
        if active_status is not None:
            intervals[active_status].append((group_start, group_end))
        
        for status, blocks in intervals.items():
            config = status_configs[status]
            for start, end in blocks:
                if start == end:
                    ax.axvline(start, ymin=config["ymin"], ymax=config["ymax"], color=config["color"], linewidth=1.2, zorder=4)
                else:
                    ax.axvspan(start, end, ymin=config["ymin"], ymax=config["ymax"], color=config["color"], alpha=0.7, edgecolor=None, zorder=4)
        
        clean_name = client_name.title().replace("Us", "US")
        ax.set_title(rf"\textbf{{{clean_name}}}")
        ax.grid(True, alpha=0.3)

        if col == 0:
            ax.set_ylabel(r"Requests Per Second")
        if row == 2:
            ax.set_xlabel(r"Time (s)")

    legend_handles = [
        plt.Line2D([0], [0], color=COLORS["green"], linewidth=1.2, label=r"\texttt{Client Load}"),
        plt.Line2D([0], [0], color=COLORS["red"], linewidth=1.2, alpha=0.3, label=r"\texttt{Others Load}"),
        mpatches.Patch(color=COLORS["pink"], alpha=0.7, label=r"\textsf{Fast-Path Block}"),
        mpatches.Patch(color=COLORS["orange"], alpha=0.7, label=r"\textsf{Regular-Path Block}"),
    ]

    fig.legend(
        handles=legend_handles,
        loc='lower center',
        bbox_to_anchor=(0.5, 1.01),
        ncol=4,
        frameon=True,
        fontsize="x-small",
        borderaxespad=0.0
    )
    
    fig.savefig(result_file, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)

plot_clients_vs_others_load(
    experiments = [
        {
            "data_dir": "../results/old10/heuristic_adaptive_localevents_no-retry_risk0.1_lr0.005_dur600_cal300_regionus",
            "clients": ["us-central1", "us-east5"]
        },
        {
            "data_dir": "../results/old10/heuristic_adaptive_localevents_no-retry_risk0.1_lr0.005_dur600_cal300_regioneurope",
            "clients": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
        }
    ],
    result_file="results/client_load_localevents_vs.pdf",
)

Client us-central1: 27 of 3205 requests dropped (0.84%)
Client us-east5: 9 of 3234 requests dropped (0.28%)
Client us-west1: 3 of 3167 requests dropped (0.09%)
Client us-west2: 3 of 3216 requests dropped (0.09%)
Client us-west4: 6 of 2810 requests dropped (0.21%)
Client europe-central2: 33 of 3074 requests dropped (1.07%)
Client europe-north2: 28 of 3232 requests dropped (0.87%)
Client europe-southwest1: 33 of 2795 requests dropped (1.18%)
Client europe-west2: 15 of 3403 requests dropped (0.44%)
Client europe-west3: 6 of 2842 requests dropped (0.21%)


In [14]:
def plot_fast_path_error_rate(experiments, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    fig_height = fig_width * 0.45

    fig, axes = plt.subplots(
        nrows=1, ncols=2, 
        sharey=True, 
        figsize=(fig_width, fig_height), 
        constrained_layout=True
    )

    region_titles = ["US Region", "Europe Region"]
    
    color_palette = [
        COLORS["orange"], 
        COLORS["green"], 
        COLORS["red"], 
        COLORS["light_blue"], 
        COLORS["pink"], 
        COLORS["dark_blue"]
    ]
    markers = ['o', 's', '^', 'D', 'v', 'p']

    unique_servers = []
    for exp_dict in experiments:
        for config in exp_dict.values():
            for s in config["servers"]:
                if s not in unique_servers:
                    unique_servers.append(s)

    server_color_map = {
        server: color_palette[i % len(color_palette)] 
        for i, server in enumerate(unique_servers)
    }
    server_marker_map = {
        server: markers[i % len(markers)] 
        for i, server in enumerate(unique_servers)
    }

    legend_handles = []
    legend_labels = []

    for idx, (exp_dict, ax) in enumerate(zip(experiments, axes)):
        server_data = {}

        for target_risk, config in sorted(exp_dict.items()):
            data_dir = config["data_dir"]
            target_servers = set(config["servers"])

            df = parse_fast_path_logs(data_dir, cutoff_mins=8)

            if df.empty:
                continue

            last_entries = df.groupby('server').last().reset_index()

            for _, row in last_entries.iterrows():
                server = row['server']

                if server in target_servers:
                    total = row['total_attempts']
                    errors = row['errors']

                    actual_error_rate = (errors / total) if total > 0 else 0.0

                    if server not in server_data:
                        server_data[server] = {'x': [], 'y': []}

                    server_data[server]['x'].append(target_risk)
                    server_data[server]['y'].append(actual_error_rate)

        all_targets = sorted(exp_dict.keys())
        max_target = max(all_targets) if all_targets else 0.5
        
        target_line, = ax.plot(
            [0, max_target], [0, max_target], 
            color="gray", 
            linestyle="--", 
            linewidth=1.2, 
            label=r"\texttt{Target (Ideal)}"
        )

        if r"\texttt{Target (Ideal)}" not in legend_labels:
            legend_handles.append(target_line)
            legend_labels.append(r"\texttt{Target (Ideal)}")

        for server, points in server_data.items():
            color = server_color_map[server]
            marker = server_marker_map[server]
            latex_label = rf"\texttt{{{server}}}"

            line, = ax.plot(
                points['x'], 
                points['y'], 
                marker=marker, 
                linestyle='-', 
                linewidth=1.2,
                markersize=4, 
                color=color,
                label=latex_label
            )

            if latex_label not in legend_labels:
                legend_handles.append(line)
                legend_labels.append(latex_label)

        title_text = region_titles[idx] if idx < len(region_titles) else f"Region {idx+1}"
        ax.set_title(rf"\textbf{{{title_text}}}")
        ax.set_xlabel(r"Target Fast-Path Error Rate")
        if idx == 0:
            ax.set_ylabel(r"Actual Fast-Path Error Rate")

        ax.set_xlim(-0.02, max_target + 0.03)
        ax.set_ylim(-0.02, max_target + 0.03)
        ax.grid(True, alpha=0.3)

    fig.legend(
        legend_handles, legend_labels, 
        loc='lower center', 
        bbox_to_anchor=(0.5, 1.01), 
        ncol=4, 
        frameon=True, 
        fontsize="x-small",
        borderaxespad=0.0
    )

    fig.savefig(result_file, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)

plot_fast_path_error_rate(
    experiments = 
    [
        {
            0.01:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.01_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.05:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.05_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.1:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.2:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.2_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.3:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.3_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.4:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.4_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
            0.5:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.5_lr0.005_dur600_cal300_regionus",
                    "servers": ["us-central1", "us-east5"]
                },
        },
        {
            0.01:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.01_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.05:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.05_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.1:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.1_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.2:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.2_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.3:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.3_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.4:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.4_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
            0.5:
                {
                    "data_dir": "../results/old10/crc_adaptive_localevents_retry_risk0.5_lr0.005_dur600_cal300_regioneurope",
                    "servers": ["europe-north2", "europe-southwest1", "europe-west2", "europe-central2"]
                },
        },
    ],
    result_file="results/server_fast_path_stats.pdf",
)

In [11]:
def plot_latency_bars(version_dirs, cutoffs, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    fig_height = fig_width * 1.4 
    
    version_frames = []
    
    for version_name, path in version_dirs.items():
        cutoff_mins = cutoffs.get(version_name, 0)
        df = process_client_files(data_dir=path, cutoff_mins=cutoff_mins)
        df["version"] = version_name
        version_frames.append(df)
            
    compiled_df = pd.concat(version_frames, ignore_index=True).sort_values(by=["client"])

    estimators_config = [
        ("Mean", "mean"),
        ("Median", "median"),
        ("P10", lambda x: np.percentile(x, 10, method="higher")),
        ("P90", lambda x: np.percentile(x, 90, method="higher")),
        ("P95", lambda x: np.percentile(x, 95, method="higher")),
        ("P99", lambda x: np.percentile(x, 99, method="higher")),
    ]

    custom_palette = {
        "Regular Paxos": "#22c55e",
        "Fast Paxos": "#f97316",
        "CRC(0.1)": "#38bdf8",
        "CRC(0.05)": "#2563eb",
        "CRC(0.01)": "#1e3a8a",
    }

    fig, axes = plt.subplots(
        nrows=3, ncols=2, sharey=True,
        figsize=(fig_width, fig_height), 
        constrained_layout=True
    )

    fig.set_constrained_layout_pads(w_pad=1/72, h_pad=1/72, hspace=0.02, wspace=0.02)
    for (est_name, est_func), ax in zip(estimators_config, axes.flat):
        sns.barplot(
            data=compiled_df,
            x='client',
            y='latency',
            hue='version',
            hue_order=list(version_dirs.keys()),
            estimator=est_func,
            errorbar=None,
            ax=ax,
            palette=custom_palette,
            edgecolor="black",
            linewidth=0.8
        )
        
        ax.set_title(fr"\textbf{{{est_name} Latency}}")
        ax.set_ylabel("Latency (ms)")
        ax.set_xlabel("")
        ax.grid(True, alpha=0.2)
        ax.tick_params(axis='x', labelsize=7)
        
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, 
        loc='lower center', 
        bbox_to_anchor=(0.5, 1.01), 
        ncol=5,  # Fits all 5 items on a single row
        frameon=True, 
        fontsize="x-small",
        borderaxespad=0.0
    )

    fig.savefig(result_file, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)

In [23]:
plot_latency_bars(
    version_dirs={
        "Regular Paxos": "../results/normal_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "Fast Paxos": "../results/fast_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "CRC(0.1)": "../results/crc_adaptive_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "CRC(0.05)": "../results/crc_adaptive_localevents_retry_risk0.05_lr0.005_dur600_cal300_regionus",
        "CRC(0.01)": "../results/crc_adaptive_localevents_retry_risk0.01_lr0.005_dur600_cal300_regionus",
    },
    cutoffs={
        "Regular Paxos": 1,
        "Fast Paxos": 1,
        "CRC(0.1)": 6,
        "CRC(0.05)": 6,
        "CRC(0.01)": 6,
    },
    result_file="results/server_version_latency_comparison_localevents_retry_us.pdf"
)

Client us-central1: 0 of 2884 requests dropped (0.00%)
Client us-east5: 0 of 2852 requests dropped (0.00%)
Client us-west1: 0 of 2810 requests dropped (0.00%)
Client us-west2: 0 of 2648 requests dropped (0.00%)
Client us-west4: 0 of 2900 requests dropped (0.00%)
Client us-central1: 0 of 2781 requests dropped (0.00%)
Client us-east5: 0 of 2430 requests dropped (0.00%)
Client us-west1: 0 of 2903 requests dropped (0.00%)
Client us-west2: 0 of 3177 requests dropped (0.00%)
Client us-west4: 0 of 2805 requests dropped (0.00%)
Client us-central1: 0 of 2654 requests dropped (0.00%)
Client us-east5: 0 of 2913 requests dropped (0.00%)
Client us-west1: 0 of 3382 requests dropped (0.00%)
Client us-west2: 0 of 3393 requests dropped (0.00%)
Client us-west4: 0 of 2491 requests dropped (0.00%)
Client us-central1: 0 of 3049 requests dropped (0.00%)
Client us-east5: 0 of 3366 requests dropped (0.00%)
Client us-west1: 0 of 3029 requests dropped (0.00%)
Client us-west2: 0 of 2688 requests dropped (0.00%)


In [4]:
def plot_version_latency_violins(version_dirs, cutoffs, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    fig_height = fig_width * 0.618
    
    version_frames = []
    
    for version_name, path in version_dirs.items():
        cutoff_mins = cutoffs.get(version_name, 0)
        df = process_client_files(data_dir=path, cutoff_mins=cutoff_mins)
        df["version"] = version_name
        version_frames.append(df)
            
    compiled_df = pd.concat(version_frames, ignore_index=True).sort_values(by=["client"])


    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height),
        constrained_layout=True
    )
    
    sns.violinplot(
        data=compiled_df,
        x='client',
        y='latency',
        hue='version',
        inner=None,
        hue_order=list(version_dirs.keys()),
        ax=ax,
        palette="muted",
    )
    
    ax.set_title(r"\textbf{Latency by Client and Server Version}")
    ax.set_ylabel(r"Latency (ms)")
    ax.set_xlabel("")
    ax.grid(True, alpha=0.3)
    
    ax.legend(
        title="Version", 
        frameon=True,
        fontsize="small",
        title_fontsize="small",
        handletextpad=0.4,
        columnspacing=0.8
    )
    
    fig.savefig(result_file, bbox_inches="tight")
    plt.close(fig)

In [10]:
def plot_version_latency_boxplots(version_dirs, cutoffs, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    fig_height = fig_width * 0.618
    
    version_frames = []
    
    for version_name, path in version_dirs.items():
        cutoff_mins = cutoffs.get(version_name, 0)
        df = process_client_files(data_dir=path, cutoff_mins=cutoff_mins)
        df["version"] = version_name
        version_frames.append(df)
            
    compiled_df = pd.concat(version_frames, ignore_index=True).sort_values(by=["client"])

    fig, ax = plt.subplots(
        figsize=(fig_width, fig_height),
        constrained_layout=True
    )
    
    sns.boxplot(
        data=compiled_df,
        x='client',
        y='latency',
        hue='version',
        hue_order=list(version_dirs.keys()),
        ax=ax,
        palette="muted",
        medianprops={"color": "red", "linewidth": 1},
        showfliers=False,
    )
    
    ax.set_title(r"\textbf{Latency by Client and Server Version}")
    ax.set_ylabel(r"Latency (ms)")
    ax.set_xlabel("")
    ax.grid(True, alpha=0.3)
    
    ax.legend(
        title="Version", 
        frameon=True,
        fontsize="small",
        title_fontsize="small",
        handletextpad=0.4,
        columnspacing=0.8
    )
    
    fig.savefig(result_file, bbox_inches="tight")
    plt.close(fig)

In [7]:
def plot_version_latency_boxplots_status(version_dirs, cutoffs, result_file):
    fig_width = LATEX_TEXTWIDTH_PT * PT_TO_INCH
    base_subplot_height = fig_width * 0.618 
    
    version_frames = []
    
    for version_name, path in version_dirs.items():
        cutoff_mins = cutoffs.get(version_name, 0)
        df = process_client_files(data_dir=path, cutoff_mins=cutoff_mins)
        df["version"] = version_name
        version_frames.append(df)
            
    compiled_df = pd.concat(version_frames, ignore_index=True).sort_values(by=["client"])
    
    
    # Filter out empty modes or "Dropped" (inf latency) requests since boxplots can't render inf properly
    valid_plot_df = compiled_df[
        (~compiled_df['accept_status'].isin(['', 'Dropped', 'Unknown'])) & 
        (compiled_df['latency'] != np.inf)
    ]
    
    modes = ["LeaderAccept", "FastAccept", "SlowAccept"]
    n_modes = len(modes)
    

    # Scale the overall figure height by the number of subplots needed
    fig, axes = plt.subplots(
        nrows=n_modes, 
        ncols=1,
        figsize=(fig_width, base_subplot_height * n_modes),
        constrained_layout=True,
        sharex=True
    )
    
    if n_modes == 1:
        axes = [axes]
        
    for ax, mode in zip(axes, modes):
        mode_df = valid_plot_df[valid_plot_df['accept_status'] == mode]
        
        sns.boxplot(
            data=mode_df,
            x='client',
            y='latency',
            hue='version',
            hue_order=list(version_dirs.keys()),
            ax=ax,
            palette="muted",
            medianprops={"color": "red", "linewidth": 1},
            showfliers = False,
        )
        
        # Add the specific mode to the title
        ax.set_title(rf"\textbf{{Latency by Client and Server Version ({mode})}}")
        ax.set_ylabel(r"Latency (ms)")
        ax.set_xlabel("")
        ax.grid(True, alpha=0.3)
        
        # Only show the legend on the first subplot to prevent clutter
        if ax != axes[0]:
            if ax.get_legend() is not None:
                ax.get_legend().remove()
        else:
            ax.legend(
                title="Version", 
                frameon=True,
                fontsize="small",
                title_fontsize="small",
                handletextpad=0.4,
                columnspacing=0.8
            )
    
    fig.savefig(result_file, bbox_inches="tight")
    plt.close(fig)

In [9]:
plot_version_latency_boxplots_status(
    version_dirs={
        "Regular Paxos": "../results/normal_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "Fast Paxos": "../results/fast_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "CRC(0.1)": "../results/crc_adaptive_localevents_retry_risk0.1_lr0.005_dur600_cal300_regionus",
        "CRC(0.05)": "../results/crc_adaptive_localevents_retry_risk0.05_lr0.005_dur600_cal300_regionus",
        "CRC(0.01)": "../results/crc_adaptive_localevents_retry_risk0.01_lr0.005_dur600_cal300_regionus",
    },
    cutoffs={
        "Regular Paxos": 1,
        "Fast Paxos": 1,
        "CRC(0.1)": 6,
        "CRC(0.05)": 6,
        "CRC(0.01)": 6,
    },
    result_file="results/server_version_boxplot_status_localevents_retry_boxplot_status_us.pdf"
)

Client us-central1: 0 of 2884 requests dropped (0.00%)
Client us-east5: 0 of 2852 requests dropped (0.00%)
Client us-west1: 0 of 2810 requests dropped (0.00%)
Client us-west2: 0 of 2648 requests dropped (0.00%)
Client us-west4: 0 of 2900 requests dropped (0.00%)
Client us-central1: 0 of 2781 requests dropped (0.00%)
Client us-east5: 0 of 2430 requests dropped (0.00%)
Client us-west1: 0 of 2903 requests dropped (0.00%)
Client us-west2: 0 of 3177 requests dropped (0.00%)
Client us-west4: 0 of 2805 requests dropped (0.00%)
Client us-central1: 0 of 2950 requests dropped (0.00%)
Client us-east5: 0 of 2896 requests dropped (0.00%)
Client us-west1: 0 of 3079 requests dropped (0.00%)
Client us-west2: 0 of 2630 requests dropped (0.00%)
Client us-west4: 0 of 2832 requests dropped (0.00%)
Client us-central1: 0 of 2749 requests dropped (0.00%)
Client us-east5: 0 of 2997 requests dropped (0.00%)
Client us-west1: 0 of 3086 requests dropped (0.00%)
Client us-west2: 0 of 3105 requests dropped (0.00%)
